# VWAP_V1.3 交易执行器（零基础详细说明）

你现在看到的是一个“教学版”的 VWAP（成交量加权平均价）交易执行系统。它不会直接连真实交易所，而是先用 `MockExchange` 模拟行情与成交，然后用你写的执行逻辑跑一遍，把每笔订单的成交/未成交、告警与尾盘强制完成都记录成 `JSONL`。

为了方便你从零开始理解，我下面会按“从上到下”的思路讲清楚：
1. 整个项目由哪些模块构成、每个模块做什么。
2. 配置文件里每个参数是什么意思、怎么用。
3. 程序如何把总 Notional 拆成多笔、如何生成限价、如何计算未成交比例并告警。
4. 现货（Spot）与期货永续（Perp）在代码里如何完全分离。
5. 日志文件 `execution_logs.jsonl` 的每个字段是什么意思。
6. 你要接真实交易所时需要改哪里。

---

你可以把它当成：
- 一个“自动下单执行器”（核心逻辑）
- 一个“风控监控器”（未成交比例/滑点/尾盘风险）
- 一个“记录器”（每笔订单的明细都落盘）

所有关键数值都尽量做成了可配置，不硬编码在逻辑里。

## 1. 你应该怎么运行它（先跑通）

本项目提供了一个演示入口脚本：`run.py`。

### 1.1 直接运行（Mock 模拟）

在项目根目录（`VWAP_V1.3`）执行：

```bash
python3 run.py --start-offset-seconds 1
```

含义是：把 `start_time` 往后推 1 秒，保证演示立刻开始。

运行结束后会生成：
- `execution_logs.jsonl`（如果示例配置里指定了输出路径）

你在终端也会看到 `ExecutionSummary`。

### 1.2 如果你想让 start_time 跟配置文件一致

默认演示模式会用“当前时间 + offset”覆盖配置里的 `start_time`，避免你一运行就等很久。

如果你希望严格遵守配置文件里的 `start_time`，可以加：

```bash
python3 run.py --respect-config-start-time --start-offset-seconds 0
```

注意：如果你配置的 `start_time` 在未来较远的时间，你的程序会在等待调度点期间运行很久，这是正常现象。

In [1]:
# 读取并预览 execution_logs.jsonl（运行 run.py 后再执行）
import json

path = 'execution_logs.jsonl'

with open(path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

print('log lines:', len(lines))

# 打印前 3 行看格式
for i in range(min(3, len(lines))):
    print('--- line', i)
    print(lines[i].strip())


log lines: 6
--- line 0
{"type": "order", "sub_order_index": 0, "sub_order_time": "2026-04-08T09:13:13.412476", "order_id": "spot-BTCUSDT-0-c2bd5997-1", "symbol": "BTCUSDT", "side": "BUY", "order_type": "LIMIT", "notional": 2000.0, "limit_price": 65017.67706923158, "avg_fill_price": 64944.52658062333, "ordered_notional": 2000.0, "filled_notional": 2000.0, "filled_qty": 0.030795512806104812, "unfilled_notional": 0.0, "unfilled_ratio": 0.0, "slippage_ratio": 0.0011250861597278546, "triggered_alarm": false, "alarm_type": null, "alarm_message": null, "raw": {"best_bid": 65017.18822569928, "best_ask": 65037.18822569928, "estimated_margin": null}}
--- line 1
{"type": "order", "sub_order_index": 1, "sub_order_time": "2026-04-08T09:13:15.412476", "order_id": "spot-BTCUSDT-1-d30b6e65-2", "symbol": "BTCUSDT", "side": "BUY", "order_type": "LIMIT", "notional": 2000.0, "limit_price": 64881.231276058716, "avg_fill_price": 64846.8183384083, "ordered_notional": 2000.0, "filled_notional": 2000.0, "fill

## 2. 项目结构（文件夹/文件分别在做什么）

你的项目根目录主要有这些部分：

- `run.py`
- `config/`（示例配置文件）
- `vwap_executor/`（真正实现 VWAP 执行器的代码）
- `vwap_executor/tests/`（少量单元测试）
- `secrets/`（交易所 API Key 占位说明）

下面按“你需要理解的层次”讲 `vwap_executor/` 里的每个模块。

---

### 2.1 `vwap_executor/__init__.py`

作用：把 `VwapConfig`、`VwapExecutionEngine` 暴露出来，方便你在 `run.py` 里直接导入。

---

### 2.2 `vwap_executor/config.py`（配置加载与参数定义）

这里定义了所有“输入参数”的数据结构。

- `CommonInput`：通用输入（标的 `symbol`、方向 `side`、总 Notional、开始时间 `start_time`、计价/基础币种）
- `ExecutionParams`：执行参数（总时长、间隔、限价偏移、滑点阈值、未成交告警阈值、尾盘强制完成规则、现货截断策略、期货杠杆等）
- `VwapConfig`：把上面两类拼起来，并增加 `instrument_type`（`spot` 或 `perp`）以及日志/告警配置。

关键点：
1. 你给什么策略，就可以在配置文件里改什么。
2. 代码里尽量不写死这些数。

---

### 2.3 `vwap_executor/models.py`（数据结构：订单、成交、告警、日志条目）

这里定义了程序内部使用的“对象类型”。最重要的几个是：

- `SubOrderSpec`：每一笔子订单的计划（子订单索引、计划时间、该子订单的目标 Notional）
- `OrderFill`：一次下单的结果（订单 id、成交时间、订单类型 LIMIT/MARKET、方向、ordered_notional、filled_notional、平均成交价等）
- `Alert`：告警对象（告警类型、时间、符号、订单 id、消息、未成交比例/剩余未成交等）
- `OrderLogEntry`：要写进 `execution_logs.jsonl` 的“明细记录”

---

### 2.4 `vwap_executor/order_manager.py`（拆单与限价价格生成）

这是 VWAP 的核心“数学/规则”模块。

它包含：
- `_split_notional_equal(total, n_slices)`：把总 Notional 平均拆成 `n_slices` 笔，最后一笔修正浮点误差，保证求和尽量等于总额。
- `build_vwap_schedule(start_time, total_duration_seconds, order_interval_seconds)`：生成每笔子订单的计划时间。
- `compute_limit_price(side, best, price_offset, ...)`：根据盘口 best bid/ask + `price_offset` 计算限价。
- `build_sub_orders(...)`：把计划时间与拆分后的每笔 Notional 组合成 `SubOrderSpec` 列表。

---

### 2.5 `vwap_executor/risk.py`（风控与告警规则）

负责三类风险监控：
- 未成交比例告警：`unfilled_notional / ordered_notional > unfilled_alarm_threshold`
- 滑点阈值告警：`slippage_ratio > max_slippage`
- 尾盘风险：尾盘强制完成前，剩余未成交占初始总 Notional 比例过大（`tail_risk_threshold_ratio`）

---

### 2.6 `vwap_executor/logging_store.py`（把订单明细落盘）

用 JSONL（每行一个 JSON）记录：
- 每笔订单的 `OrderLogEntry`
- 每条告警的 `Alert`

输出文件默认由配置 `log_storage.output_jsonl_path` 控制。

---

### 2.7 `vwap_executor/exchange/`（交易所接口与 Mock 实现）

这一层的设计思想是：
- 你的 VWAP 执行器只关心“下单”和“拿盘口”，不关心交易所细节。
- 交易所适配器实现 `BaseExchange` 里规定的方法。

文件包含：
- `base.py`：定义抽象接口 `BaseExchange`
- `mock.py`：用随机游走价格 + 简化撮合，模拟 LIMIT 部分成交和 MARKET 全成交（带尾盘冲击滑点）。
- `credentials.py`：交易所 API Key 的预留位置（通过环境变量读取：`EXCHANGE_API_KEY` / `EXCHANGE_API_SECRET`；你可以参考 `secrets/.env.example`；当前演示用 Mock，因此 key 不强制需要）。

---

### 2.8 `vwap_executor/executors/`（现货/期货分离执行器）

这是“现货与期货必须完全分离”的关键。

文件包含：
- `base_executor.py`：基类，封装了共同的 VWAP 执行框架：拆单、按时间等待、限价下单、计算未成交比例、告警、尾盘强制市价。
- `spot_executor.py`：现货专用约束：
  - `SELL` 不能超卖。
  - 不足持仓时可配置为截断到可卖范围（默认 `spot_sell_truncate_to_holdings=true`）。
- `perp_executor.py`：期货专用逻辑：
  - 期货输入是名义 Notional。
  - 杠杆倍数 `leverage` 用于计算保证金占用：`margin = notional / leverage`。
  - 在日志里把每笔 `estimated_margin` 写进去，满足你“杠杆与 Notional 处理可追溯”的需求。
（补充）`vwap_executor/scheduler.py`：预留的时间调度器/时间提供器。
你可以理解为：如果未来你要把执行调度从“简单等待 until”升级成更精确/可测试的调度系统，这个文件就会用上。
当前教学版主要用基类里的 `await _wait_until(...)` 来做时间等待，因此它不会影响你运行。

---

### 2.9 `vwap_executor/execution_engine.py`（选择正确的执行器）

这是把一切串起来的入口。

做的事很简单：
- 根据 `config.instrument_type` 选择：`SpotVwapExecutor` 或 `PerpVwapExecutor`
- 创建 `RiskManager`
- 调用执行器的 `execute()`

---

### 2.10 `run.py`（程序真正启动位置）

你运行的脚本。

它会：
- 读取 `config/example_config.json`（或你指定的 config）
- 根据配置的 `exchange.adapter` 启动 `MockExchange` 或真实交易所适配器
- 创建 `VwapExecutionEngine` 并执行
- 最后把日志落盘到 `execution_logs.jsonl`

## 3. 一次执行到底发生了什么（从代码看流程）

以 `base_executor.py` 为主线，你可以按下面步骤理解。

---

### Step 1：生成“子订单计划”（拆单）

代码在基类里通过：
- `build_vwap_schedule(...)` 生成每笔子订单的计划时间
- `build_sub_orders(...)` 把总 Notional 平均拆成多笔，每笔得到 `target_notional`

例子（对应你的需求描述）：
- total_duration = 20 分钟
- interval = 1 分钟
=> 约 20 笔子订单
- 每笔约 total/20

---

### Step 2：等到计划时间点（按时间调度）

基类用 `await self._wait_until(spec.scheduled_time)` 等到该时间再执行这一笔。

说明：
- 在真实交易里你需要更精确的调度。
- 在这个教学版里，`MockExchange` 的价格更新不强依赖系统时钟，因此只要能触发到循环即可。

---

### Step 3：拿盘口 best bid/ask，并计算限价

基类调用：
- `exchange.get_best_prices(symbol)` 得到 best bid/ask
- `compute_limit_price(side, best, price_offset, ...)`

限价规则：
- BUY：`limit = ask * (1 + price_offset)`
- SELL：`limit = bid * (1 - price_offset)`

---

### Step 4：下单并得到成交（由 Spot/Perp executor 决定约束）

基类调用 `_submit_single_limit(...)`，但具体逻辑由子类实现：
- `spot_executor.py`：
  - SELL 时检查 `get_available_base_qty`，不足则截断（默认允许截断）
  - BUY 不会受超卖约束影响
- `perp_executor.py`：
  - 不检查现货持仓
  - 根据 `leverage` 计算保证金并写入 `fill.estimated_margin`

---

### Step 5：计算未成交比例并触发告警

一笔子订单下单后：
- `unfilled_notional = ordered_notional - filled_notional`
- `unfilled_ratio = unfilled_notional / ordered_notional`

当 `unfilled_ratio > unfilled_alarm_threshold` 时触发告警。

同时还会检查滑点：
- 如果 `slippage_ratio > max_slippage`，也会触发订单级告警。

---

### Step 6：尾盘强制完成（市价一次性完成剩余）

在循环进入尾盘时：
- 当剩余时间 `<= tail_force_early_seconds`（你的配置）就触发尾盘。
- 使用一次市价单尝试把剩余未成交补齐。

尾盘风险告警：
- 在执行尾盘强制市价之前，检查剩余未成交金额占初始总金额比例是否 > `tail_risk_threshold_ratio`。

---

### Step 7：把明细和告警写入 JSONL

每一笔子订单都会生成 `OrderLogEntry` 并写入 `execution_logs.jsonl`：
- 下单时间
- 下单价格（限价/市价的 limit_price 不同）
- 下单方向
- 每笔 Notional（ordered_notional）
- 成交数量/成交均价
- 未成交金额/未成交比例
- 是否触发告警
- 以及 raw 附加字段（例如 best bid/ask、期货 estimated_margin）

告警也会作为 JSONL 里的另一种类型写入。

## 4. 配置文件 `config/example_config.json` 怎么读（零基础版）

你可以把它想成“策略作业单”。你要什么，就在这里填。

项目默认使用它：
- `instrument_type`: `spot` 或 `perp`
- `common`：标的、方向、总 Notional、开始时间、币种
- `execution`：所有执行参数（拆单、限价偏移、阈值、尾盘规则、杠杆等）
- `log_storage.output_jsonl_path`：输出日志文件路径

---

### 4.1 你最需要理解的 execution 字段

- `total_duration_seconds`：总共执行多久（例如 600 秒=10 分钟）
- `order_interval_seconds`：每隔多久下一个子订单（例如每 2 秒一次）
- `price_offset`：限价偏移（相对比例）。
- BUY：越大则 limit 越靠近/高于 ask；SELL 同理。
- `max_slippage`：滑点上限，用于触发告警
- `unfilled_alarm_threshold`：未成交比例超过多少触发告警（默认 0.10=10%）
- `tail_force_early_seconds`：到尾盘提前多久触发尾盘市价强制完成（默认 0 表示严格到每个子订单时间点判断）
- `tail_risk_threshold_ratio`：尾盘风险阈值（默认 0.10=10%）
- `tail_market_slippage`：尾盘市价额外冲击滑点（模拟用）
- `spot_sell_truncate_to_holdings`：现货 SELL 不允许超卖，不足时是否截断到可卖
- `leverage`：期货杠杆倍数。保证金占用：`margin = notional / leverage`

---

### 4.2 其他字段

- `mock_fill_sensitivity`：Mock 撮合“更容易成交/更难成交”的程度（教学用）。
- `debug`：预留字段（当前示例未把它大量用于输出）。

## 5. 交易明细（`execution_logs.jsonl`）字段怎么理解

日志文件是 JSONL：每一行是一条记录。
记录里有 `type`：
- `type = "order"`：订单明细
- `type = "alert"`：告警

以 `type=order` 的记录为例（你可以在运行后对照）：

- `sub_order_index`：第几笔子订单（尾盘强制市价会用 `-1`）
- `sub_order_time`：该子订单计划时间
- `order_id`：由执行器生成/交易所返回的订单 id
- `symbol`：标的
- `side`：BUY/SELL
- `order_type`：LIMIT 或 MARKET
- `notional`：**这一笔订单的 Notional（ordered_notional）**
- `limit_price`：限价订单会有值；市价订单为 `null`
- `avg_fill_price`：平均成交价
- `ordered_notional`：订单下单时的名义金额
- `filled_notional`：实际成交的名义金额
- `filled_qty`：模拟成交数量（由 `qty = notional / price` 推导）
- `unfilled_notional`：未成交金额
- `unfilled_ratio`：未成交比例
- `slippage_ratio`：滑点比例（仅限价有意义；市价一般为 null）
- `triggered_alarm`：是否触发告警（未成交或滑点任意一个都可能触发）
- `alarm_type` / `alarm_message`：告警类型与消息文本
- `raw`：附加信息
- spot：包含 best bid/ask
- perp：还会包含估算的 `estimated_margin` 等

---

零基础小提醒：
- “未成交金额”是相对于这一笔订单的 ordered_notional。
- “未成交比例”就是：未成交金额 / ordered_notional。
- 尾盘市价的目的：把“剩余未成交金额”尽量一次性补齐。

## 6. 现货 vs 期货：两套完全分离的执行逻辑

这个项目满足你的需求：现货与期货在架构上做了独立模块。

你可以直接看：
- `vwap_executor/executors/spot_executor.py`
- `vwap_executor/executors/perp_executor.py`

### 6.1 现货（Spot）规则：禁止卖空（不允许负仓位）

当执行方向是 `SELL` 时：
- 先通过 `exchange.get_available_base_qty(symbol)` 拿到“当前可卖数量”。
- 如果按 limit_price 换算出来的目标 Notional 超过可卖范围，就截断到可卖 Notional。

因此：
- 默认不会出现负仓位。
- 如果你希望“不足就报错而不是截断”，把配置里的 `spot_sell_truncate_to_holdings=false`。

### 6.2 期货（Perp）规则：Notional + 杠杆计算保证金

期货输入的是名义 Notional。

保证金占用计算（在本代码里用于日志追溯）：
- `margin = notional / leverage`

做多/做空都支持：
- `BUY` 或 `SELL` 都会走同一个 Perp 执行器。

当前版本声明仅支持 `U 本位永续`（也就是 `USDT_perpetual`）。

---

### 你如何接真实交易所（最重要）

你需要做的是：
1. 新写一个交易所适配器，实现 `vwap_executor/exchange/base.py` 的 `BaseExchange` 接口。
2. 用你的适配器替换 `run.py` 里创建 `MockExchange` 的那一行。
3. 在适配器里把 `place_limit_order / place_market_order / get_best_prices / get_available_base_qty` 对接到真实交易所 API。

交易所 API Key 预留在：
- `vwap_executor/exchange/credentials.py`

这里示例用环境变量读取：
- `EXCHANGE_API_KEY`
- `EXCHANGE_API_SECRET`

但如果你在配置里选择的是 `exchange.adapter = mock`，就会使用 `MockExchange`，所以 key 不需要也能跑。

接真实交易所时，你把 key 用上即可。

## 7. 单元测试与下一步扩展

测试文件在：`vwap_executor/tests/test_core.py`。

你可以运行：

```bash
python3 -m unittest -q
```

目前测试覆盖：
- 拆分 Notional 求和是否正确
- 未成交比例告警是否能触发

---

如果你想把这个教学版升级成“可用的实盘系统”，通常下一步会包括：
1. 实现真实交易所适配器（WebSocket/REST 盘口、订单回报、成交查询）。
2. 增加“订单状态跟踪”（真实 limit order 可能不会立刻全部成交）。
3. 更精细的尾盘触发逻辑与撤单重发策略。
4. 风险控制扩展（例如最大下单偏离、最大订单数量、交易所拥堵处理等）。

如果你愿意，我也可以按你的目标交易所（Binance/OKX/Bybit 之类）把 `BaseExchange` 适配器补齐。